# train

> One epoch of training, one epoch of validation, atomic checkpointing, and a resumable training loop with a phased loss schedule and differential learning rates.

The loop is generic: pass `num_classes` and it trains a binary or multi-class model with no other changes. Targets are integer-label `(B, H, W)` tensors throughout; the loss and metrics modules adapt internally.

In [ ]:
#| default_exp train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import json
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn


In [ ]:
#| exporti
from segmentation_tools_pt.channels import DEFAULT_CHANNELS
from segmentation_tools_pt.losses import SegLoss, resolve_loss_weights
from segmentation_tools_pt.metrics import SegMetrics
from segmentation_tools_pt.model import get_param_groups, freeze_encoder, unfreeze_encoder
from segmentation_tools_pt.visualize import (
    save_color_dim_png, save_near_zero_png, save_overlay_grid_jpeg, save_multilabel_overlay_grid_jpeg,
    rank_samples, rank_samples_per_class, save_error_poster,
)


## Training callbacks

Optional hooks at train / epoch / batch granularity. Subclass `TrainCallback` and pass a list to `train()` — the default loop is unchanged when `callbacks=None`.

Set `state.stop_training = True` in a callback (e.g. early stopping) to exit the epoch loop cleanly after the current epoch.

## One training epoch

One job: run the loader once, update the model, and return averaged losses and training metrics. Mixed precision is used when a `GradScaler` is supplied.

In [ ]:
#| export
@dataclass
class TrainState:
    """Mutable context passed to every training callback hook."""
    model: nn.Module
    device: torch.device
    num_epochs: int
    num_classes: int = 1
    epoch: int = 0
    batch_idx: int = 0
    optimizer: Optional[torch.optim.Optimizer] = None
    scheduler: Optional[Any] = None
    scaler: Optional[Any] = None
    criterion: Optional[SegLoss] = None
    train_loader: Any = None
    val_loader: Any = None
    batch: Optional[dict] = None
    logits: Optional[torch.Tensor] = None
    loss_dict: Optional[Dict[str, Any]] = None
    train_metrics: Optional[Dict[str, float]] = None
    val_metrics: Optional[Dict[str, float]] = None
    history: List[dict] = field(default_factory=list)
    best_score: float = 0.0
    best_metric: str = 'iou'
    checkpoint_dir: Optional[str] = None
    is_best: bool = False
    stop_training: bool = False


In [ ]:
#| export

class TrainCallback:
    """Base callback; override only the hooks you need. Lower `order` runs first."""
    order: int = 0

    def on_train_begin(self, state: TrainState) -> None: pass
    def on_train_end(self, state: TrainState) -> None: pass
    def on_epoch_begin(self, state: TrainState) -> None: pass
    def on_epoch_end(self, state: TrainState) -> None: pass
    def on_train_batch_begin(self, state: TrainState) -> None: pass
    def on_train_batch_end(self, state: TrainState) -> None: pass
    def on_val_begin(self, state: TrainState) -> None: pass
    def on_val_end(self, state: TrainState) -> None: pass
    def on_val_batch_begin(self, state: TrainState) -> None: pass
    def on_val_batch_end(self, state: TrainState) -> None: pass
    def on_checkpoint(self, state: TrainState, path: str) -> None: pass



## Default monitoring callbacks

fastai-style activation histograms (`ActivationStatsCallback`) and YOLO-style validation overlay JPEGs (`PredOverlayCallback`), both on by default. Plus the opt-in post-training `ErrorAnalysisCallback` (issue #18): when enabled it ranks the worst samples per split at **train end** and writes 4-column JPEG posters (image | GT overlay α=0.1 | pred overlay | FN/FP difference) plus `top_ranked.json` under `checkpoints/error_analysis/` — multilabel gets one poster per class head with `per_class=True`. Default **off** so existing runs are unchanged. Use `build_default_callbacks()` from Hydra or pass directly to `train()`.

In [ ]:
#| export
_HOOK_TYPES = (nn.Conv2d, nn.BatchNorm2d)

def _hookable_layers(model: nn.Module, max_layers: int) -> List[Tuple[str, nn.Module]]:
    "Up to `max_layers` hook targets, evenly spaced across Conv/BN modules."
    layers = [(n, m) for n, m in model.named_modules() if isinstance(m, _HOOK_TYPES) and n]
    if not layers:
        layers = [(n, m) for n, m in model.named_modules() if n]
    if not layers and isinstance(model, _HOOK_TYPES):
        # bare Conv2d/BatchNorm2d used directly as the whole model (e.g. tests) has no
        # named submodules at all -- named_modules() yields only ('', model).
        layers = [(type(model).__name__, model)]
    if len(layers) <= max_layers:
        return layers
    idxs = np.linspace(0, len(layers) - 1, max_layers, dtype=int)
    return [layers[i] for i in idxs]

class ActivationStatsCallback(TrainCallback):
    """Record per-layer activation histograms; save fastai colorful-dimension PNGs each epoch."""
    order = 10

    def __init__(
        self,
        checkpoint_dir: str,
        every: int = 1,
        n_bins: int = 40,
        hist_range: Tuple[float, float] = (0.0, 10.0),
        max_layers: int = 6,
    ):
        self.root = Path(checkpoint_dir) / 'activation_stats'
        self.every = every
        self.n_bins = n_bins
        self.hmin, self.hmax = hist_range
        self.max_layers = max_layers
        self._hooks: List[Any] = []
        self._layer_names: List[str] = []
        self._epoch_hists: List[List[torch.Tensor]] = []
        self._epoch_near_zero: List[List[float]] = []
        self._near_zero_history: Dict[str, List[Tuple[int, float]]] = {}
        self._pending: List[Optional[Tuple[torch.Tensor, float]]] = []

    def on_train_begin(self, state: TrainState) -> None:
        layers = _hookable_layers(state.model, self.max_layers)
        self._layer_names = [n.replace('.', '_') for n, _ in layers]
        self._epoch_hists = [[] for _ in layers]
        self._epoch_near_zero = [[] for _ in layers]
        self._near_zero_history = {name: [] for name in self._layer_names}
        self._pending = [None] * len(layers)

        def _make_hook(li: int):
            def hook(module, _inp, out):
                if not module.training:
                    return
                o = out.detach().float()
                if o.numel() == 0:
                    return
                flat = o.flatten()
                hist = torch.histc(flat, self.n_bins, self.hmin, self.hmax).cpu()
                nz = float((flat <= 0.05).float().mean().item())
                self._pending[li] = (hist, nz)
            return hook

        for i, (_, mod) in enumerate(layers):
            self._hooks.append(mod.register_forward_hook(_make_hook(i)))

    def on_train_batch_end(self, state: TrainState) -> None:
        if state.batch_idx % self.every != 0:
            return
        for i, item in enumerate(self._pending):
            if item is None:
                continue
            hist, nz = item
            self._epoch_hists[i].append(hist)
            self._epoch_near_zero[i].append(nz)
            self._pending[i] = None

    def on_epoch_end(self, state: TrainState) -> None:
        epoch_dir = self.root / f'epoch{state.epoch + 1:03d}'
        epoch_dir.mkdir(parents=True, exist_ok=True)
        for i, name in enumerate(self._layer_names):
            hists = self._epoch_hists[i]
            if not hists:
                continue
            stack = torch.stack(hists)
            save_color_dim_png(stack.T, epoch_dir / f'layer{i:02d}_{name}_color_dim.png', title=name)
            mean_nz = float(np.mean(self._epoch_near_zero[i]))
            self._near_zero_history[name].append((state.epoch, mean_nz))
            ep, vals = zip(*self._near_zero_history[name])
            save_near_zero_png(list(ep), list(vals), epoch_dir / f'layer{i:02d}_{name}_near_zero.png', title=name)
        self._epoch_hists = [[] for _ in self._layer_names]
        self._epoch_near_zero = [[] for _ in self._layer_names]

    def on_train_end(self, state: TrainState) -> None:
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

class PredOverlayCallback(TrainCallback):
    """Save YOLO-style validation overlay JPEG mosaics during training.

    `task='multilabel'` uses `visualize.save_multilabel_overlay_grid_jpeg`
    (per-channel colored GT-vs-Pred panels) instead of the single-label
    `save_overlay_grid_jpeg`.
    """
    order = 20

    def __init__(
        self,
        checkpoint_dir: str,
        threshold=0.5,
        every: int = 1,
        max_val_batches: int = 3,
        jpeg_quality: int = 90,
        num_classes: int = 1,
        task: str = None,   # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
        class_names: Optional[Sequence[str]] = None,  # multilabel legend labels
    ):
        self.root = Path(checkpoint_dir) / 'predictions'
        self.threshold = threshold
        self.every = every
        self.max_val_batches = max_val_batches
        self.jpeg_quality = jpeg_quality
        self.num_classes = num_classes
        self.task = task or ('binary' if num_classes == 1 else 'multiclass')
        self.class_names = list(class_names) if class_names is not None else None

    def _decode_logits(self, logits: torch.Tensor) -> torch.Tensor:
        if self.num_classes == 1:
            return (torch.sigmoid(logits[:, 0]) >= self.threshold).long()
        return logits.argmax(dim=1)

    def on_val_batch_end(self, state: TrainState) -> None:
        if state.epoch % self.every != 0:
            return
        if state.batch_idx >= self.max_val_batches:
            return
        if state.batch is None or state.logits is None:
            return
        out_dir = self.root / f'epoch{state.epoch + 1:03d}'
        out_dir.mkdir(parents=True, exist_ok=True)
        images, masks = state.batch['image'], state.batch['mask']
        bi = state.batch_idx
        if self.task == 'multilabel':
            probs = torch.sigmoid(state.logits)
            save_multilabel_overlay_grid_jpeg(
                images, masks, probs,
                out_dir / f'val_batch{bi}_multilabel.jpg',
                class_names=self.class_names,
                threshold=self.threshold,
                jpeg_quality=self.jpeg_quality,
            )
            return
        preds = self._decode_logits(state.logits)
        for mode, suffix in (('labels', 'labels'), ('pred', 'pred'), ('compare', 'compare')):
            save_overlay_grid_jpeg(
                images, masks, preds,
                out_dir / f'val_batch{bi}_{suffix}.jpg',
                threshold=self.threshold,
                mode=mode,
                jpeg_quality=self.jpeg_quality,
            )

def build_default_callbacks(
    checkpoint_dir: str,
    threshold=0.5,
    activation_stats: bool = True,
    pred_overlays: bool = True,
    num_classes: int = 1,
    act_every: int = 1,
    act_n_bins: int = 40,
    act_hist_range: Tuple[float, float] = (0.0, 10.0),
    act_max_layers: int = 6,
    pred_every: int = 1,
    pred_max_val_batches: int = 3,
    pred_jpeg_quality: int = 90,
    task: str = None,   # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
    class_names: Optional[Sequence[str]] = None,  # multilabel legend labels for PredOverlayCallback
    error_analysis: bool = False,  # post-training worst-sample posters; default off so existing runs are unchanged
    ea_top_k: int = 12,            # worst samples per split (and per class)
    ea_rank_by: str = 'loss',      # 'loss' | 'iou' | 'dice' | 'recall' | 'precision'
    ea_splits: Sequence[str] = ('train', 'val'),
    ea_alpha: float = 0.1,         # GT / pred overlay transparency
    ea_jpeg_quality: int = 90,
    ea_per_class: bool = True,     # multilabel: one poster per class head
    drift_check: bool = False,     # verify encoder weights unchanged after training; default off so existing runs are unchanged
    drift_expect_frozen: Optional[bool] = None,  # True = raise on drift (freeze-forever); None = caller decides (scripts derive from freeze_epochs)
) -> List[TrainCallback]:
    "Factory for on-by-default training monitoring callbacks."
    cbs: List[TrainCallback] = []
    if activation_stats:
        cbs.append(ActivationStatsCallback(
            checkpoint_dir, every=act_every, n_bins=act_n_bins,
            hist_range=act_hist_range, max_layers=act_max_layers,
        ))
    if pred_overlays:
        cbs.append(PredOverlayCallback(
            checkpoint_dir, threshold=threshold, every=pred_every,
            max_val_batches=pred_max_val_batches, jpeg_quality=pred_jpeg_quality,
            num_classes=num_classes, task=task, class_names=class_names,
        ))
    if error_analysis:
        cbs.append(ErrorAnalysisCallback(
            checkpoint_dir, top_k=ea_top_k, rank_by=ea_rank_by, splits=ea_splits,
            alpha=ea_alpha, jpeg_quality=ea_jpeg_quality, per_class=ea_per_class,
            num_classes=num_classes, task=task, class_names=class_names, threshold=threshold,
        ))
    if drift_check:
        cbs.append(EncoderDriftCheck(expect_frozen=bool(drift_expect_frozen)))
    return cbs


class ErrorAnalysisCallback(TrainCallback):
    """Post-training error triage: rank the worst samples per split and save 4-column JPEG posters.

    One row per sample: image | GT overlay (`alpha`) | pred overlay | difference overlay
    (FN green / FP red / overlap magenta). `rank_by='loss'` ranks highest loss first;
    pixel metrics (`iou`/`dice`/`recall`/`precision`) rank lowest first. Multilabel with
    `per_class=True` writes one poster per class head. Outputs under
    `{checkpoint_dir}/error_analysis/`: `{split}_top{k}.jpg` (+ `_{class}` per class)
    and `top_ranked.json` (`sample_id`, `split`, `score`, `rank_by`, optional `class`).
    """
    order = 30

    def __init__(
        self,
        checkpoint_dir: str,
        top_k: int = 12,               # worst samples per split (and per class)
        rank_by: str = 'loss',         # 'loss' | 'iou' | 'dice' | 'recall' | 'precision'
        splits: Sequence[str] = ('train', 'val'),
        alpha: float = 0.1,            # GT / pred overlay transparency
        jpeg_quality: int = 90,
        per_class: bool = True,        # multilabel: one poster per class head
        num_classes: int = 1,
        task: str = None,              # 'binary' | 'multiclass' | 'multilabel'; None infers
        class_names: Optional[Sequence[str]] = None,
        threshold=0.5,
    ):
        self.root = Path(checkpoint_dir) / 'error_analysis'
        self.top_k = top_k
        self.rank_by = rank_by
        self.splits = list(splits)
        self.alpha = alpha
        self.jpeg_quality = jpeg_quality
        self.per_class = per_class
        self.num_classes = num_classes
        self.task = task or ('binary' if num_classes == 1 else 'multiclass')
        self.class_names = list(class_names) if class_names is not None else None
        self.threshold = threshold

    def _record(self, split: str, rank: int, rec: dict, class_name: Optional[str]) -> dict:
        r = {'split': split, 'rank': rank, 'sample_id': rec['sample_id'],
             'score': rec['score'], 'rank_by': self.rank_by}
        if class_name is not None:
            r['class'] = class_name
        return r

    def on_train_end(self, state: TrainState) -> None:
        criterion = state.criterion
        if criterion is None:
            return
        self.root.mkdir(parents=True, exist_ok=True)
        loaders = {'train': state.train_loader, 'val': state.val_loader}
        ranked_all = []
        for split in self.splits:
            loader = loaders.get(split)
            if loader is None:
                continue
            ds, model = loader.dataset, state.model
            if self.per_class and self.task == 'multilabel':
                for c in range(self.num_classes):
                    name = self.class_names[c] if self.class_names and c < len(self.class_names) else f'class{c}'
                    ranked = rank_samples_per_class(model, ds, criterion, state.device, c,
                                                    rank_by=self.rank_by, threshold=self.threshold)
                    save_error_poster(
                        model, ds, ranked, state.device, self.root / f'{split}_top{self.top_k}_{name}.jpg',
                        rank_by=self.rank_by, k=self.top_k, alpha=self.alpha, threshold=self.threshold,
                        jpeg_quality=self.jpeg_quality, task='multilabel', class_idx=c,
                        class_names=self.class_names)
                    ranked_all += [self._record(split, i, rec, name) for i, rec in enumerate(ranked[:self.top_k])]
            else:
                ranked = rank_samples(model, ds, criterion, state.device, rank_by=self.rank_by,
                                      threshold=self.threshold, task=self.task)
                save_error_poster(
                    model, ds, ranked, state.device, self.root / f'{split}_top{self.top_k}.jpg',
                    rank_by=self.rank_by, k=self.top_k, alpha=self.alpha, threshold=self.threshold,
                    jpeg_quality=self.jpeg_quality, task=self.task, class_names=self.class_names)
                ranked_all += [self._record(split, i, rec, None) for i, rec in enumerate(ranked[:self.top_k])]
        (self.root / 'top_ranked.json').write_text(json.dumps(ranked_all, indent=2))


def _sd_digest(sd: Dict[str, torch.Tensor]) -> str:
    "Stable digest of a state dict: sorted keys + float32 bytes (numpy has no bf16 dtype)."
    h = hashlib.md5()
    for k, v in sorted(sd.items()):
        h.update(k.encode())
        h.update(v.detach().cpu().contiguous().float().numpy().tobytes())
    return h.hexdigest()


class EncoderDriftCheck(TrainCallback):
    """Post-training answer to "did the encoder weights change?": digest the encoder
    state dict at `on_train_begin`, compare at `on_train_end`.

    Bit-identical -> prints a confirmation (the guarantee behind freeze-forever,
    since AdamW never updates params without grads). Changed -> prints per-layer
    cosine similarity, worst first. With `expect_frozen=True` a changed encoder
    raises RuntimeError; otherwise it is a drift report for trainable encoders.
    """
    order = 40  # after error analysis (30): the guard verdict is the last word

    def __init__(self, expect_frozen: bool = False, max_report: int = 8):
        self.expect_frozen, self.max_report = expect_frozen, max_report
        self.before: Optional[Dict[str, torch.Tensor]] = None
        self.before_digest: Optional[str] = None

    def on_train_begin(self, state):
        enc = getattr(state.model, 'encoder', None)
        if enc is None:
            print('EncoderDriftCheck: model has no .encoder; nothing to check.')
            return
        self.before = {k: v.detach().clone() for k, v in enc.state_dict().items()}
        self.before_digest = _sd_digest(self.before)

    def on_train_end(self, state):
        if self.before is None:
            return
        enc = state.model.encoder
        if _sd_digest(enc.state_dict()) == self.before_digest:
            print('EncoderDriftCheck: encoder weights bit-identical after training.')
            self.before = None
            return
        sims = {}
        for k, v in enc.state_dict().items():
            b = self.before.get(k)
            if b is None or b.numel() != v.numel():
                continue
            sims[k] = torch.cosine_similarity(
                v.detach().flatten().float(), b.flatten().float(), dim=0).item()
        worst = sorted(sims, key=sims.get)[:self.max_report]
        print('EncoderDriftCheck: encoder weights CHANGED; worst layers by cosine similarity:')
        for k in worst:
            print(f'  {k}: cos={sims[k]:.4f}')
        if self.expect_frozen:
            raise RuntimeError('EncoderDriftCheck: encoder drifted although expect_frozen=True '
                               '(e.g. freeze_epochs: -1) — frozen encoders must be bit-identical.')
        self.before = None


## Encoder drift check

After training — especially a DINOv3 freeze-forever run (`freeze_epochs: -1`, issue #16) — we want a trustworthy answer to *did the encoder weights change?*

- **Deterministic (ground truth):** digest the encoder state dict before and after; bit-identical means unchanged. AdamW never updates params without grads, so a frozen encoder must match exactly.
- **Quantitative:** when drift is expected (trainable encoder), per-layer cosine similarity initial vs final shows *where* and *how much*.
- **NOT activation histograms:** those shift every epoch because augmentation changes the inputs, not because weights moved — a health signal, not a frozen-ness check.

Plan:

1. Verify the digest is stable and change-sensitive, and that cosine similarity reports drift magnitude.
2. Export `_sd_digest` + `EncoderDriftCheck` (order 40, fires after error analysis) and wire `build_default_callbacks(..., drift_check=..., drift_expect_frozen=...)` — default off so existing runs are unchanged (same pattern as `error_analysis`).
3. Test: frozen run passes; drifted run reports, raises under `expect_frozen`; models without `.encoder` are a no-op.

In [ ]:
# experiment: digest is stable + change-sensitive; cosine similarity reports drift magnitude
sd = {'w': torch.randn(8, 4), 'b': torch.zeros(8)}
d0 = _sd_digest(sd)
sd2 = {'w': sd['w'].clone(), 'b': sd['b'].clone()}
sd2['w'][0, 0] += 1e-3
assert _sd_digest(sd) == d0                      # digest stable across calls
assert _sd_digest(sd2) != d0                     # detects a tiny perturbation
sim = torch.cosine_similarity(sd['w'].flatten(), sd2['w'].flatten(), dim=0).item()
print(f'perturbed weight cosine similarity: {sim:.6f}')
assert 0.99 < sim <= 1.0


In [ ]:
#| export
def _sorted_callbacks(callbacks: Optional[Sequence[TrainCallback]]) -> List[TrainCallback]:
    if not callbacks:
        return []
    return sorted(callbacks, key=lambda cb: getattr(cb, 'order', 0))


def _call_callbacks(
    callbacks: Optional[Sequence[TrainCallback]],
    hook: str,
    state: Optional[TrainState],
    **kwargs,
) -> None:
    if not callbacks or state is None:
        return
    for cb in _sorted_callbacks(callbacks):
        getattr(cb, hook)(state, **kwargs)


In [ ]:
#| export
def train_one_epoch(
    model: nn.Module,                  # model returning (B, C, H, W) logits
    loader,                            # training DataLoader of {'image','mask'} batches
    criterion: SegLoss,                # loss returning a breakdown dict
    optimizer: torch.optim.Optimizer,  # optimizer
    device: torch.device,              # cpu or cuda
    scaler: Optional[object] = None,   # torch.amp.GradScaler for AMP, or None
    grad_clip: float = 1.0,            # max gradient norm
    num_classes: int = 1,              # 1 = binary, >1 = multi-class or multilabel
    callbacks: Optional[Sequence[TrainCallback]] = None,
    state: Optional[TrainState] = None,
    task: str = None,                  # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
    metrics_threshold=0.5,             # float, or per-channel Sequence[float] for task='multilabel'
    batch_augs: Optional[dict] = None, # {'mosaic','mixup','cutmix','copy_paste'}: probs; None = off
) -> Dict[str, float]:
    """Train for one epoch; return averaged losses and `train_*` metrics."""
    model.train()
    running = {'loss': 0., 'loss_tversky': 0., 'loss_dice': 0., 'loss_boundary': 0.}
    metrics = SegMetrics(num_classes=num_classes, task=task, threshold=metrics_threshold)
    for batch_idx, batch in enumerate(loader):
        if state is not None:
            state.batch_idx = batch_idx
            state.batch = batch
        _call_callbacks(callbacks, 'on_train_batch_begin', state)
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['mask'].to(device, non_blocking=True)
        if batch_augs:
            images, targets, _ = apply_batch_augs(images, targets, batch_augs)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(images)
                ld = criterion(logits, targets)
            scaler.scale(ld['loss']).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(images)
            ld = criterion(logits, targets)
            ld['loss'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        if state is not None:
            state.logits = logits.detach()
            state.loss_dict = {k: float(v) for k, v in ld.items()}
        _call_callbacks(callbacks, 'on_train_batch_end', state)
        for k in running:
            running[k] += float(ld[k])
        metrics.update(logits.detach(), targets)
    n = len(loader)
    out = {k: v / n for k, v in running.items()}
    out.update({f'train_{k}': v for k, v in metrics.compute().items() if k != 'per_class'})
    return out


## Validation

In [ ]:
#| export
@torch.no_grad()
def validate(
    model: nn.Module,    # model returning logits
    loader,              # validation DataLoader
    criterion: SegLoss,  # loss for tracking val loss
    device: torch.device,
    num_classes: int = 1,
    callbacks: Optional[Sequence[TrainCallback]] = None,
    state: Optional[TrainState] = None,
    task: str = None,                  # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
    metrics_threshold=0.5,             # float, or per-channel Sequence[float] for task='multilabel'
) -> Dict[str, float]:
    """Run validation; return overlap metrics plus `val_loss`."""
    model.eval()
    _call_callbacks(callbacks, 'on_val_begin', state)
    metrics = SegMetrics(num_classes=num_classes, task=task, threshold=metrics_threshold)
    running = 0.0
    for batch_idx, batch in enumerate(loader):
        if state is not None:
            state.batch_idx = batch_idx
            state.batch = batch
        _call_callbacks(callbacks, 'on_val_batch_begin', state)
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['mask'].to(device, non_blocking=True)
        logits = model(images)
        ld = criterion(logits, targets)
        running += float(ld['loss'])
        if state is not None:
            state.logits = logits
            state.loss_dict = {k: float(v) for k, v in ld.items()}
        _call_callbacks(callbacks, 'on_val_batch_end', state)
        metrics.update(logits, targets)
    res = {k: v for k, v in metrics.compute().items() if k != 'per_class'}
    res['val_loss'] = running / len(loader)
    if state is not None:
        state.val_metrics = res
    _call_callbacks(callbacks, 'on_val_end', state)
    return res


## Atomic checkpointing

Write to a temp file then rename, so an interrupted save never corrupts the last good checkpoint.

In [ ]:
#| export
def save_checkpoint(
    path,                       # destination .pth path
    epoch: int, model, optimizer, scheduler, scaler,  # training state
    best_score: float, history: list,                 # bookkeeping
) -> None:
    "Atomically save full training state."
    ckpt = dict(epoch=epoch, model_state=model.state_dict(), optimizer_state=optimizer.state_dict(),
                scheduler_state=scheduler.state_dict(),
                scaler_state=scaler.state_dict() if scaler else None,
                best_score=best_score, history=history)
    tmp = str(path) + '.tmp'
    torch.save(ckpt, tmp)
    Path(tmp).replace(path)

def load_checkpoint(path, model, optimizer, scheduler, scaler, device):
    "Restore training state; return `(start_epoch, best_score, history)`."
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    if scaler and ckpt.get('scaler_state'):
        scaler.load_state_dict(ckpt['scaler_state'])
    return ckpt['epoch'] + 1, ckpt['best_score'], ckpt['history']

## Batch-level augmentations (YOLO-style pick-one)

Sample-level augs (`get_train_transforms`) act inside the DataLoader; **batch-level** augs act on the
whole batch right after it lands on the device, in `train_one_epoch`. Four types, all GPU tensors,
all shape-generic over mask layout (`(B,H,W)`, `(B,1,H,W)`, `(B,K,H,W)`):

| Type | What it does |
|---|---|
| `mosaic` | per-sample 2x2 canvas (self + 3 rolled donors) at `(2H, 2W)`, random crop back to `(H, W)`; no-op when `B < 4` |
| `mixup` | convex blend with a permuted partner; scalar `lam ~ Beta(32, 32)` (concentrated near 0/1 → mild blending, masks stay mostly intact) |
| `cutmix` | one random box (image + mask) swapped with a permuted donor |
| `copy_paste` | paste a horizontally-flipped donor's foreground (image + mask union) into each sample |

Wiring (issue #17):

- Config lives under `augment.batch.*`: `enabled` (default **false**) plus per-type probs
  (default **0**). Master off, or master on with all probs 0, must be a strict no-op — training
  is bit-identical to today.
- **Pick-one**: per batch, draw `u ~ U(0,1)` and walk cumulative thresholds in a fixed order —
  each type fires with exactly its configured marginal probability and **at most one type is
  applied per batch** (no MixUp-then-CutMix stacking).
- All randomness flows through one Python `random` instance (seeded by `set_seed`), so runs stay
  reproducible; donor permutations included.
- Metrics/loss see the augmented targets — same convention as YOLO. Validation is untouched.

Plan: verify pick-one marginals + each aug on synthetic batches → export `pick_batch_aug`,
`apply_*_batch`, `apply_batch_augs` → wire `batch_augs` through `train_one_epoch` and `train()`
→ Hydra resolver `resolve_batch_aug_cfg` (nbs/15) → configs.

In [ ]:
# experiment: pick-one marginals + each aug on synthetic batches (before promoting to export)
import random
import torch

BATCH_AUG_ORDER = ('mosaic', 'mixup', 'cutmix', 'copy_paste')

def pick_batch_aug(probs, rng=None):
    rng = rng or random
    cum, u = 0.0, rng.random()
    for name in BATCH_AUG_ORDER:
        cum += float(probs.get(name, 0.0))
        if u < cum:
            return name
    return None

N = 50_000
probs = {'mosaic': 0.2, 'mixup': 0.3, 'cutmix': 0.1, 'copy_paste': 0.25}
rng = random.Random(42)
counts = {k: 0 for k in probs}; counts[None] = 0
for _ in range(N):
    counts[pick_batch_aug(probs, rng)] += 1
for k, p in probs.items():
    emp = counts[k] / N
    assert abs(emp - p) < 0.01, (k, emp, p)
print('marginals:', {k: round(v / N, 4) for k, v in counts.items()})
assert abs(sum(pick_batch_aug({'mixup': 0.5}, rng) is not None for _ in range(N)) / N - 0.5) < 0.01
assert pick_batch_aug({k: 0.0 for k in probs}, rng) is None
print('pick-one: single-type marginal + all-zero no-op OK')

In [ ]:
#| export
_BATCH_AUG_ORDER = ('mosaic', 'mixup', 'cutmix', 'copy_paste')

def _perm_from_rng(rng, B: int, device) -> torch.Tensor:
    "Batch permutation drawn from the seeded Python `random` instance (keeps runs reproducible)."
    return torch.as_tensor(rng.sample(range(B), B), device=device)

def pick_batch_aug(
    probs: dict,   # per-type probabilities, e.g. {'mixup': 0.05, 'mosaic': 0.1}
    rng=None,      # python-random-like RNG; None = module-level `random`
) -> Optional[str]:  # one of ('mosaic','mixup','cutmix','copy_paste') or None
    """YOLO-style pick-one: at most one batch-aug type per batch, each with marginal prob `probs[type]`.

    Example:
        >>> pick_batch_aug({'mixup': 1.0}, random.Random(0))
        'mixup'
    """
    rng = rng or random
    cum, u = 0.0, rng.random()
    for name in _BATCH_AUG_ORDER:
        cum += float(probs.get(name, 0.0))
        if u < cum:
            return name
    return None

def apply_mixup_batch(images: torch.Tensor, targets: torch.Tensor, rng=None):
    """Blend each sample with a permuted partner (image + mask); scalar `lam ~ Beta(32, 32)` = mild blending."""
    rng = rng or random
    lam = min(0.999, max(0.001, rng.betavariate(32, 32)))
    perm = _perm_from_rng(rng, images.shape[0], images.device)
    images = lam * images + (1 - lam) * images[perm]
    targets = lam * targets + (1 - lam) * targets[perm]
    return images, targets

def apply_cutmix_batch(images: torch.Tensor, targets: torch.Tensor, rng=None):
    """Swap one random box (image + mask) with a permuted donor sample."""
    rng = rng or random
    _, _, H, W = images.shape
    lam = rng.betavariate(1.0, 1.0)
    cut_h = max(1, int(H * (1 - lam) ** 0.5)); cut_w = max(1, int(W * (1 - lam) ** 0.5))
    cy, cx = rng.randrange(H), rng.randrange(W)
    y0, y1 = max(0, cy - cut_h // 2), min(H, cy - cut_h // 2 + cut_h)
    x0, x1 = max(0, cx - cut_w // 2), min(W, cx - cut_w // 2 + cut_w)
    perm = _perm_from_rng(rng, images.shape[0], images.device)
    donor_img, donor_tg = images[perm], targets[perm]  # advanced indexing -> copies, no aliasing
    images = images.clone()
    images[:, :, y0:y1, x0:x1] = donor_img[:, :, y0:y1, x0:x1]
    targets = targets.clone()
    targets[..., y0:y1, x0:x1] = donor_tg[..., y0:y1, x0:x1]
    return images, targets

def apply_mosaic_batch(images: torch.Tensor, targets: torch.Tensor, rng=None):
    """Per-sample 2x2 mosaic (self + 3 rolled donors) on a (2H, 2W) canvas, random crop back to (H, W).

    No-op when the batch has fewer than 4 samples.
    """
    rng = rng or random
    B, _, H, W = images.shape
    if B < 4:
        return images, targets
    d1, d2, d3 = images.roll(1, 0), images.roll(2, 0), images.roll(3, 0)
    m1, m2, m3 = targets.roll(1, 0), targets.roll(2, 0), targets.roll(3, 0)
    top_i, bot_i = torch.cat([images, d1], dim=-1), torch.cat([d2, d3], dim=-1)
    top_m, bot_m = torch.cat([targets, m1], dim=-1), torch.cat([m2, m3], dim=-1)
    canvas_img, canvas_tg = torch.cat([top_i, bot_i], dim=-2), torch.cat([top_m, bot_m], dim=-2)
    y0, x0 = rng.randrange(0, H + 1), rng.randrange(0, W + 1)
    return canvas_img[:, :, y0:y0 + H, x0:x0 + W], canvas_tg[..., y0:y0 + H, x0:x0 + W]

def apply_copy_paste_batch(images: torch.Tensor, targets: torch.Tensor, rng=None):
    """Paste a horizontally-flipped donor's foreground (image + mask union) into each sample.

    Works for binary `(B,1,H,W)`/`(B,H,W)` and multilabel `(B,K,H,W)` masks; the paste
    location is any donor foreground pixel, masks merge via maximum.
    """
    rng = rng or random
    perm = _perm_from_rng(rng, images.shape[0], images.device)
    donor_img = torch.flip(images[perm], dims=[-1])
    donor_tg = torch.flip(targets[perm], dims=[-1])
    fg = donor_tg > 0
    if fg.dim() > 3:            # (B,1,H,W) or (B,K,H,W): reduce channel axis -> (B,1,H,W)
        paste = fg.amax(dim=1, keepdim=True)
    else:                        # (B,H,W)
        paste = fg.unsqueeze(1)
    images = torch.where(paste, donor_img, images)
    targets = torch.maximum(targets, donor_tg)
    return images, targets

_BATCH_AUG_FNS = {
    'mosaic': apply_mosaic_batch, 'mixup': apply_mixup_batch,
    'cutmix': apply_cutmix_batch, 'copy_paste': apply_copy_paste_batch,
}

def apply_batch_augs(
    images: torch.Tensor, targets: torch.Tensor,
    batch_augs: Optional[dict],  # {'mosaic','mixup','cutmix','copy_paste'}: probs (extra keys ignored); None = off
    rng=None,
):
    """Apply at most one batch-level augmentation per batch; return `(images, targets, applied_name)`.

    `batch_augs=None` or all-zero probs -> identity with `applied_name=None` (strict no-op).
    """
    rng = rng or random
    if not batch_augs:
        return images, targets, None
    name = pick_batch_aug(batch_augs, rng)
    fn = _BATCH_AUG_FNS.get(name)
    if fn is None:
        return images, targets, None
    images, targets = fn(images, targets, rng=rng)
    return images, targets, name

In [ ]:
# test: pick-one — every type fires with its configured marginal prob; at most one per batch
from collections import Counter

probs = {'mosaic': 0.2, 'mixup': 0.3, 'cutmix': 0.1, 'copy_paste': 0.25}
N = 20_000
rng = random.Random(42)
counts = Counter(pick_batch_aug(probs, rng) for _ in range(N))
for k, p in probs.items():
    assert abs(counts[k] / N - p) < 0.02, (k, counts[k] / N, p)
assert abs((counts[None] / N) - (1 - sum(probs.values()))) < 0.02
assert sum(counts.values()) == N  # every draw is exactly one type or none

rng = random.Random(1)
assert abs(sum(pick_batch_aug({'mixup': 0.5}, rng) is not None for _ in range(N)) / N - 0.5) < 0.02
assert pick_batch_aug({'mosaic': 0.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0}, rng) is None
print('pick_batch_aug marginals + no-op OK')

In [ ]:
# test: each batch aug on synthetic batches — shapes, semantics, dispatcher on/off/pick-one
B, C, H, W = 8, 3, 64, 64
images = torch.rand(B, C, H, W)
targets = (torch.rand(B, 1, H, W) > 0.8).float()
targets[:, :, 10:20, 10:20] = 1.0  # guaranteed donor foreground

im2, tg2 = apply_mixup_batch(images, targets, random.Random(0))
assert im2.shape == images.shape and tg2.shape == targets.shape
assert bool((im2 >= images.min() - 1e-6).all()) and bool((im2 <= images.max() + 1e-6).all())  # fp tolerance

im2, tg2 = apply_cutmix_batch(images, targets, random.Random(0))
assert im2.shape == images.shape and not torch.equal(im2, images)

im2, tg2 = apply_mosaic_batch(images, targets, random.Random(0))
assert im2.shape == (B, C, H, W) and not torch.equal(im2, images)
assert torch.equal(apply_mosaic_batch(images[:3], targets[:3], random.Random(0))[0], images[:3])  # B<4 no-op

rng = random.Random(7)
im2, tg2 = apply_copy_paste_batch(images, targets, rng)
perm = torch.as_tensor(random.Random(7).sample(range(B), B), device=images.device)
donor_img, donor_tg = torch.flip(images[perm], dims=[-1]), torch.flip(targets[perm], dims=[-1])
assert torch.equal(tg2, torch.maximum(targets, donor_tg))
paste = (donor_tg > 0).expand_as(images)  # (B,1,H,W) broadcasts over channels
assert torch.allclose(im2[paste], donor_img[paste])
keep = ((targets > 0) & ~(donor_tg > 0)).expand_as(images)
assert torch.allclose(im2[keep], images[keep])

im3, _, name = apply_batch_augs(images, targets, None)
assert name is None and torch.equal(im3, images)
zero = {'mosaic': 0.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0}
im3, _, name = apply_batch_augs(images, targets, zero)
assert name is None and torch.equal(im3, images)
for p, want in [({'mosaic': 1.0}, 'mosaic'), ({'mixup': 1.0}, 'mixup'),
                ({'cutmix': 1.0}, 'cutmix'), ({'copy_paste': 1.0}, 'copy_paste')]:
    _, _, name = apply_batch_augs(images, targets, p, rng=random.Random(0))
    assert name == want, (p, name)

# (B, H, W) long masks (direct-dataset style) and multilabel (B, K, H, W) flow through
long_tg = (torch.rand(B, H, W) > 0.8).long()
im4, tg4, name = apply_batch_augs(images, long_tg, {'copy_paste': 1.0}, rng=random.Random(0))
assert tg4.shape == long_tg.shape and tg4.dtype == long_tg.dtype
ml_tg = (torch.rand(B, 3, H, W) > 0.7).float()
im5, tg5, name = apply_batch_augs(images, ml_tg, {'cutmix': 1.0}, rng=random.Random(0))
assert tg5.shape == ml_tg.shape and name == 'cutmix'
print('batch aug behaviors + dispatcher OK')

In [ ]:
# test: train_one_epoch + train() wiring — off == all-zero probs (identical numbers); on runs pick-one
from torch.utils.data import Dataset, DataLoader

class _Toy(Dataset):
    def __len__(self): return 8
    def __getitem__(self, i):
        return {'image': torch.rand(3, 32, 32), 'mask': (torch.rand(32, 32) > 0.5).long(), 'sample_id': str(i)}

loader = DataLoader(_Toy(), batch_size=4, shuffle=False)
crit = SegLoss(w_tversky=0.5, w_dice=0.5, w_boundary=0.0)
dev = torch.device('cpu')

torch.manual_seed(0)
m1 = nn.Conv2d(3, 1, 3, padding=1)
opt1 = torch.optim.AdamW(m1.parameters(), lr=1e-3)
base = train_one_epoch(m1, loader, crit, opt1, dev, num_classes=1, batch_augs=None)

torch.manual_seed(0)
m2 = nn.Conv2d(3, 1, 3, padding=1)
opt2 = torch.optim.AdamW(m2.parameters(), lr=1e-3)
zero = train_one_epoch(m2, loader, crit, opt2, dev, num_classes=1,
                       batch_augs={'mosaic': 0.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0})
assert zero == base, 'all-zero batch probs must match batch_augs=None exactly'

torch.manual_seed(0)
m3 = nn.Conv2d(3, 1, 3, padding=1)
opt3 = torch.optim.AdamW(m3.parameters(), lr=1e-3)
mixed = train_one_epoch(m3, loader, crit, opt3, dev, num_classes=1,
                        batch_augs={'mosaic': 0.5, 'mixup': 0.5})
assert all(v == v for v in mixed.values())  # finite

print('train_one_epoch batch_augs wiring OK (train() wiring tested under its export below)')

## The training loop

Composes the pieces above: builds loaders, anneals the loss phase, optionally warms up with a frozen encoder, tracks the best validation mean-IoU, and writes resumable checkpoints.

### Encoder freeze semantics (`freeze_epochs`)

- `0` — never freeze; the encoder trains from epoch 0 (default).
- `N > 0` — decoder warm-up: freeze before training, unfreeze when the epoch counter reaches `N` (epochs `0..N-1` stay frozen).
- `-1` — freeze forever (issue #16): the encoder stays frozen for the entire run; only the decoder and segmentation head train. Useful when the pretrained representation should not drift — e.g. ViT-style encoders (DINOv3) or small datasets that overfit a tuned backbone.

Plan:

1. Verify the condition logic on the three cases (0 / N>0 / -1) over epoch values.
2. Change `train()` so it freezes whenever `freeze_epochs != 0`; the existing unfreeze condition `freeze_epochs > 0 and epoch == freeze_epochs` never fires for `-1` (epochs start at 0), so freeze-forever needs no extra branch.
3. Test end-to-end on a synthetic encoder/decoder/head model, tracking `requires_grad` per epoch.

In [ ]:
# experiment: verify freeze-condition semantics for 0 / N>0 / -1 on epoch values before touching train()
EPOCHS = 3
for fe in (0, 2, -1):
    frozen_at_start = fe != 0
    unfreeze_at = [e for e in range(EPOCHS) if fe > 0 and e == fe]
    print(f'freeze_epochs={fe:>2}: frozen at start={frozen_at_start}, unfreezes at epoch {unfreeze_at or "never"}')

assert [(fe != 0, [e for e in range(EPOCHS) if fe > 0 and e == fe]) for fe in (0, 2, -1)] == \
       [(False, []), (True, [2]), (True, [])]


In [ ]:
#| export
def train(
    root: str,                      # patch dataset root (None when pre-built loaders are supplied)
    model: nn.Module,               # model to train
    train_loader=None,              # pre-built DataLoader (direct mode); built from root if None
    val_loader=None,                # pre-built DataLoader (direct mode); built from root if None
    num_classes: int = 1,           # 1 = binary, >1 = multi-class or multilabel
    num_epochs: int = 60,           # total epochs
    batch_size: int = 16,           # batch size
    encoder_lr: float = 1e-4,       # encoder LR
    decoder_lr: float = 1e-3,       # decoder LR
    freeze_epochs: int = 0,         # 0 = no freeze; N > 0 = unfreeze at epoch N; -1 = frozen forever
    crop_size: int = 256,           # crop size
    n_source_channels: int = 2,     # raw channels per sample
    channel_names: Sequence[str] = DEFAULT_CHANNELS,  # model-image assembly
    mean: Optional[Sequence[float]] = None,  # per-channel mean
    std: Optional[Sequence[float]] = None,   # per-channel std
    device_str: str = 'cuda',       # device
    checkpoint_dir: str = 'checkpoints',  # where to save
    use_amp: bool = True,           # mixed precision on cuda
    save_every: int = 5,            # periodic checkpoint cadence
    resume: bool = True,            # resume from latest.pth if present
    loss_alpha: float = 0.7,        # Tversky FN weight (higher → fewer misses)
    loss_beta: float = 0.3,         # Tversky FP weight
    loss_gamma: float = 1.33,       # focal exponent
    loss_phases: Optional[list] = None,  # [{until_epoch, w_tversky, w_dice, w_boundary}]; None = fixed weights
    metrics_threshold=0.5,          # sigmoid threshold: float, or per-channel Sequence[float] for task='multilabel'
    best_metric: str = 'iou',       # metric to gate best.pth: iou | dice | recall | precision
    aug=None,                       # AugmentConfig for training augmentations
    boundary_blur: Optional[dict] = None,  # kwargs for BlurBoundaryZone (patch-mode training transform)
    batch_augs: Optional[dict] = None,     # batch-level aug probs {'mosaic','mixup','cutmix','copy_paste'}; None = off
    num_workers: int = 4,           # DataLoader worker processes
    resume_from: Optional[str] = None,  # explicit checkpoint path to resume from; None = use latest.pth
    seed: Optional[int] = None,     # run seed for reproducible sampling and worker RNGs
    callbacks: Optional[Sequence[TrainCallback]] = None,
    task: str = None,                              # 'binary' | 'multiclass' | 'multilabel'; None infers from num_classes
    mask_mode: str = 'single',                     # 'single' | 'stacked' | 'per_class_files' (patch-mode dataset only)
    mask_channels: Optional[int] = None,           # K, required for mask_mode='stacked'
    class_names: Optional[Sequence[str]] = None,   # K class name suffixes, required for mask_mode='per_class_files'
) -> list:
    """Resumable training loop with configurable loss phases; returns per-epoch history."""
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR
    from torch.utils.data import DataLoader, WeightedRandomSampler
    from segmentation_tools_pt.dataset import SegmentationDataset, _make_loader_generator, _seed_worker
    from segmentation_tools_pt.transforms import get_train_transforms, get_val_transforms

    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    device = torch.device(device_str if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # Build loaders from patch root when not supplied (patch mode)
    if train_loader is None or val_loader is None:
        c = len(channel_names)
        _mean = tuple(mean) if mean is not None else (0.,) * c
        _std  = tuple(std)  if std  is not None else (1.,) * c
        common = dict(num_classes=num_classes, n_source_channels=n_source_channels,
                      channel_names=channel_names,
                      mask_mode=mask_mode, mask_channels=mask_channels, class_names=class_names)
        train_ds = SegmentationDataset(
            root, 'train',
            transform=get_train_transforms(crop_size, _mean, _std, aug=aug, boundary_blur=boundary_blur),
            **common,
        )
        val_ds = SegmentationDataset(
            root, 'val',
            transform=get_val_transforms(crop_size, _mean, _std),
            **common,
        )
        sampler = WeightedRandomSampler(
            weights=train_ds.get_sample_weights(),
            num_samples=len(train_ds),
            replacement=True,
            generator=_make_loader_generator(seed),
        )
        pin = (device.type == 'cuda')
        worker_init_fn = _seed_worker if num_workers > 0 else None
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, sampler=sampler,
            num_workers=num_workers, pin_memory=pin, drop_last=True,
            persistent_workers=(num_workers > 0),
            worker_init_fn=worker_init_fn,
            generator=_make_loader_generator(seed),
        )
        val_loader = DataLoader(
            val_ds, batch_size=batch_size, shuffle=False,
            num_workers=num_workers, pin_memory=pin, drop_last=False,
            persistent_workers=(num_workers > 0),
            worker_init_fn=worker_init_fn,
            generator=_make_loader_generator(None if seed is None else seed + 1),
        )

    # Configure loss — initial weights from first phase
    _phases = loss_phases or [{'until_epoch': -1, 'w_tversky': 0.6, 'w_dice': 0.4, 'w_boundary': 0.0}]
    _w0 = resolve_loss_weights(0, _phases)
    criterion = SegLoss(alpha=loss_alpha, beta=loss_beta, gamma=loss_gamma,
                        w_tversky=_w0[0], w_dice=_w0[1], w_boundary=_w0[2], task=task).to(device)
    optimizer = AdamW(get_param_groups(model, encoder_lr, decoder_lr), weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda') if (use_amp and device.type == 'cuda') else None

    start_epoch, best_score, history = 0, 0.0, []
    latest = Path(checkpoint_dir) / 'latest.pth'
    if resume:
        if resume_from is not None:
            _resume_path = Path(resume_from)
        else:
            _resume_path = latest
        if _resume_path.exists():
            print(f'Resuming from {_resume_path}')
            start_epoch, best_score, history = load_checkpoint(str(_resume_path), model, optimizer, scheduler, scaler, device)
        elif resume_from is not None:
            raise FileNotFoundError(f'resume_from checkpoint not found: {_resume_path}')
    if freeze_epochs != 0:
        freeze_encoder(model)

    state = TrainState(
        model=model, device=device, num_epochs=num_epochs, num_classes=num_classes,
        optimizer=optimizer, scheduler=scheduler, scaler=scaler, criterion=criterion,
        train_loader=train_loader, val_loader=val_loader,
        history=history, best_score=best_score, best_metric=best_metric,
        checkpoint_dir=checkpoint_dir,
    )
    _call_callbacks(callbacks, 'on_train_begin', state)

    for epoch in range(start_epoch, num_epochs):
        state.epoch = epoch
        state.history = history
        state.best_score = best_score
        _call_callbacks(callbacks, 'on_epoch_begin', state)
        _w = resolve_loss_weights(epoch, _phases)
        criterion.set_weights(*_w)
        if freeze_epochs > 0 and epoch == freeze_epochs:
            unfreeze_encoder(model)  # -1 never matches (epochs start at 0) -> frozen forever
        tr = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler,
            num_classes=num_classes, callbacks=callbacks, state=state,
            task=task, metrics_threshold=metrics_threshold, batch_augs=batch_augs,
        )
        state.train_metrics = tr
        va = validate(
            model, val_loader, criterion, device, num_classes=num_classes,
            callbacks=callbacks, state=state,
            task=task, metrics_threshold=metrics_threshold,
        )
        scheduler.step()
        log = dict(epoch=epoch, train_loss=tr['loss'], val_loss=va['val_loss'],
                   val_iou=va['iou'], val_dice=va['dice'], val_recall=va['recall'],
                   lr=scheduler.get_last_lr()[0])
        history.append(log)
        (Path(checkpoint_dir) / 'history.json').write_text(json.dumps(history, indent=2))
        args = (epoch, model, optimizer, scheduler, scaler)
        is_best = va[best_metric] > best_score
        state.is_best = is_best
        print(
            f"epoch {epoch+1:03d}/{num_epochs}"
            f"  train={tr['loss']:.4f}"
            f"  val={va['val_loss']:.4f}"
            f"  iou={va['iou']:.4f}"
            f"  dice={va['dice']:.4f}"
            f"  recall={va['recall']:.4f}"
            f"  lr={scheduler.get_last_lr()[0]:.2e}"
            + ("  *best*" if is_best else ""),
            flush=True,
        )
        if is_best:
            best_score = va[best_metric]
            state.best_score = best_score
            best_path = str(Path(checkpoint_dir) / 'best.pth')
            save_checkpoint(best_path, *args, best_score, history)
            _call_callbacks(callbacks, 'on_checkpoint', state, path=best_path)
        if (epoch + 1) % save_every == 0:
            periodic_path = str(Path(checkpoint_dir) / f'periodic_ep{epoch:03d}.pth')
            save_checkpoint(periodic_path, *args, best_score, history)
            _call_callbacks(callbacks, 'on_checkpoint', state, path=periodic_path)
        latest_path = str(latest)
        save_checkpoint(latest_path, *args, best_score, history)
        _call_callbacks(callbacks, 'on_checkpoint', state, path=latest_path)
        _call_callbacks(callbacks, 'on_epoch_end', state)
        if state.stop_training:
            break
    _call_callbacks(callbacks, 'on_train_end', state)
    return history

In [ ]:
# smoke test: one epoch + validate + checkpoint round-trip + monitoring callbacks
import tempfile
from torch.utils.data import Dataset, DataLoader

class _Toy(Dataset):
    def __len__(self): return 8
    def __getitem__(self, i):
        return {'image': torch.rand(3, 16, 16), 'mask': (torch.rand(16, 16) > 0.5).long(), 'sample_id': str(i)}

class _CountBatches(TrainCallback):
    def __init__(self): self.train_batches = self.val_batches = 0
    def on_train_batch_end(self, state): self.train_batches += 1
    def on_val_batch_end(self, state): self.val_batches += 1

loader = DataLoader(_Toy(), batch_size=2)
model = nn.Conv2d(3, 1, 3, padding=1)
crit = SegLoss(w_tversky=0.5, w_dice=0.5, w_boundary=0.0)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
dev = torch.device('cpu')
st = TrainState(model=model, device=dev, num_epochs=1)
cb = _CountBatches()
tr = train_one_epoch(model, loader, crit, opt, dev, num_classes=1, callbacks=[cb], state=st)
assert 'loss' in tr and 'train_iou' in tr
assert cb.train_batches == len(loader)
va = validate(model, loader, crit, dev, num_classes=1, callbacks=[cb], state=st)
assert 'val_loss' in va and 'iou' in va
assert cb.val_batches == len(loader)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=2)
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / 'ckpt.pth'
    save_checkpoint(p, 0, model, opt, sched, None, 0.0, [])
    se, best, hist = load_checkpoint(p, model, opt, sched, None, dev)
    assert se == 1 and best == 0.0
    ckpt = Path(d) / 'checkpoints'
    mon = build_default_callbacks(str(ckpt), threshold=0.5, num_classes=1)
    hist2 = train(
        root=None, model=model, train_loader=loader, val_loader=loader,
        num_epochs=1, batch_size=2, device_str='cpu', checkpoint_dir=str(ckpt),
        use_amp=False, resume=False, callbacks=mon,
    )
    assert len(hist2) == 1
    hist3 = train(
        root=None, model=model, train_loader=loader, val_loader=loader,
        num_epochs=1, batch_size=2, device_str='cpu', checkpoint_dir=str(ckpt),
        use_amp=False, resume=False, batch_augs={'mixup': 1.0, 'cutmix': 0.5},
    )
    assert len(hist3) == 1
    assert list((ckpt / 'activation_stats').rglob('*_color_dim.png'))
    assert list((ckpt / 'predictions').rglob('val_batch0_labels.jpg'))


In [ ]:
# smoke test: multilabel (K=3) train loop end to end -- loss, metrics, and per-channel overlays all task-aware
class _ToyMultilabel(Dataset):
    def __len__(self): return 8
    def __getitem__(self, i):
        return {'image': torch.rand(3, 16, 16), 'mask': (torch.rand(3, 16, 16) > 0.5).float(), 'sample_id': str(i)}

ml_loader = DataLoader(_ToyMultilabel(), batch_size=2)
ml_model = nn.Conv2d(3, 3, 3, padding=1)
with tempfile.TemporaryDirectory() as d:
    ckpt = Path(d) / 'checkpoints'
    mon = build_default_callbacks(str(ckpt), num_classes=3, task='multilabel', pred_overlays=True,
                                   class_names=['chip', 'void', 'lifted'])
    hist = train(
        root=None, model=ml_model, train_loader=ml_loader, val_loader=ml_loader,
        num_classes=3, num_epochs=1, batch_size=2, device_str='cpu', checkpoint_dir=str(ckpt),
        use_amp=False, resume=False, task='multilabel', metrics_threshold=[0.5, 0.5, 0.5],
        callbacks=mon,
    )
    assert len(hist) == 1
    # PredOverlayCallback now renders per-channel colored GT-vs-Pred overlays for multilabel
    ml_jpgs = list((ckpt / 'predictions').rglob('*_multilabel.jpg'))
    assert ml_jpgs, 'expected multilabel overlay JPEGs'
    assert not list((ckpt / 'predictions').rglob('*_compare.jpg'))  # single-label filenames not used
    # ActivationStatsCallback has no task dependency, still runs
    assert list((ckpt / 'activation_stats').rglob('*_color_dim.png'))
print('train() ✓  multilabel: task threaded through loss/metrics, per-channel overlays rendered')

In [ ]:
# smoke test: ErrorAnalysisCallback -- default off in build_default_callbacks; binary posters for train+val when enabled
class _Toy(Dataset):
    def __len__(self): return 6
    def __getitem__(self, i):
        m = torch.zeros(16, 16, dtype=torch.long); m[4:10, 4:10] = 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': f'bin_{i}'}

bin_loader = DataLoader(_Toy(), batch_size=2)
bin_model2 = nn.Conv2d(3, 1, 3, padding=1)
with tempfile.TemporaryDirectory() as d:
    ckpt = Path(d) / 'checkpoints'
    cbs = build_default_callbacks(str(ckpt), num_classes=1)
    assert not any(isinstance(cb, ErrorAnalysisCallback) for cb in cbs), 'error analysis must stay off by default'
    mon = build_default_callbacks(str(ckpt), num_classes=1, error_analysis=True,
                                  ea_top_k=3, ea_rank_by='iou', ea_splits=('train', 'val'))
    assert sum(isinstance(cb, ErrorAnalysisCallback) for cb in mon) == 1
    hist = train(root=None, model=bin_model2, train_loader=bin_loader, val_loader=bin_loader,
                 num_epochs=1, batch_size=2, device_str='cpu', checkpoint_dir=str(ckpt),
                 use_amp=False, resume=False, callbacks=mon)
    ea = ckpt / 'error_analysis'
    assert (ea / 'train_top3.jpg').exists() and (ea / 'val_top3.jpg').exists()
    ranked = json.loads((ea / 'top_ranked.json').read_text())
    assert {r['split'] for r in ranked} == {'train', 'val'}
    assert all(r['rank_by'] == 'iou' and 'class' not in r for r in ranked)
    assert ranked[0]['score'] <= ranked[-1]['score'], 'metric ranking must be worst-first (lowest IoU first)'
print('error-analysis ✓  binary: train+val posters, worst-first JSON, default-off preserved')

In [ ]:
# smoke test: multilabel ErrorAnalysisCallback with per_class=True -- one poster per class head, no combined poster
class _ToyML2(Dataset):
    def __len__(self): return 6
    def __getitem__(self, i):
        m = torch.zeros(3, 16, 16)
        m[0, 2:8, 2:8] = 1.0; m[1, 9:14, 4:12] = 1.0; m[2] = m[1] * (torch.rand(16, 16) > 0.7)
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': f'ml_{i}'}

ml_loader2 = DataLoader(_ToyML2(), batch_size=2)
with tempfile.TemporaryDirectory() as d:
    ckpt = Path(d) / 'checkpoints'
    mon = build_default_callbacks(str(ckpt), num_classes=3, task='multilabel',
                                  class_names=['chip', 'void', 'lifted'],
                                  error_analysis=True, ea_top_k=2, ea_rank_by='loss')
    hist = train(root=None, model=ml_model, train_loader=ml_loader2, val_loader=ml_loader2,
                 num_epochs=1, batch_size=2, device_str='cpu', checkpoint_dir=str(ckpt),
                 use_amp=False, resume=False, task='multilabel', metrics_threshold=[0.5, 0.5, 0.5],
                 callbacks=mon)
    ea = ckpt / 'error_analysis'
    for name in ('chip', 'void', 'lifted'):
        assert (ea / f'val_top2_{name}.jpg').exists(), f'missing per-class poster for {name}'
    assert not (ea / 'val_top2.jpg').exists(), 'per_class=True must not write a combined poster'
    ranked = json.loads((ea / 'top_ranked.json').read_text())
    assert {r['class'] for r in ranked} == {'chip', 'void', 'lifted'}
    assert all(r['rank_by'] == 'loss' for r in ranked)
    by_split_class = {}
    for r in ranked: by_split_class.setdefault((r['split'], r['class']), []).append(r['score'])
    for (split, name), scores in by_split_class.items():
        assert len(scores) == 2 and scores[0] >= scores[-1], \
            f'{split}/{name} ranking must be worst-first (highest loss)'
print('error-analysis ✓  multilabel: per-class posters ranked per head, JSON carries class names')

In [ ]:
# smoke test: freeze semantics — freeze_epochs -1 (forever) / 0 (never) / 2 (warm-up, unfreeze at epoch 2)
class _Toy(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        return {'image': torch.rand(3, 16, 16), 'mask': (torch.rand(16, 16) > 0.5).long(), 'sample_id': str(i)}

class _EncDec(nn.Module):  # stub with .encoder so freeze_encoder / unfreeze_encoder are not no-ops
    def __init__(self):
        super().__init__()
        self.encoder = nn.Conv2d(3, 4, 3, padding=1)
        self.decoder = nn.Conv2d(4, 4, 3, padding=1)
        self.segmentation_head = nn.Conv2d(4, 1, 3, padding=1)
    def forward(self, x): return self.segmentation_head(self.decoder(self.encoder(x)))

class _TrackFreeze(TrainCallback):
    # track on the first training batch: on_epoch_begin fires *before* the unfreeze line,
    # so batch-level is the state actually used for the epoch's gradient updates
    def __init__(self, model): self.model, self.frozen, self._seen = model, [], set()
    def on_train_batch_begin(self, state):
        if state.epoch in self._seen: return
        self._seen.add(state.epoch)
        self.frozen.append(not any(p.requires_grad for p in self.model.encoder.parameters()))

loader = DataLoader(_Toy(), batch_size=2)
expected = {-1: [True, True, True], 0: [False, False, False], 2: [True, True, False]}
with tempfile.TemporaryDirectory() as d:
    for fe, exp in expected.items():
        model = _EncDec()
        cb = _TrackFreeze(model)
        train(root=None, model=model, train_loader=loader, val_loader=loader,
              num_epochs=3, batch_size=2, freeze_epochs=fe, device_str='cpu',
              checkpoint_dir=str(Path(d) / f'fe{fe}'), use_amp=False, resume=False,
              callbacks=[cb])
        print(f'freeze_epochs={fe:>2}: frozen per epoch = {cb.frozen}')
        assert cb.frozen == exp, (fe, cb.frozen, exp)


In [ ]:
# smoke test: EncoderDriftCheck — wiring in build_default_callbacks + frozen guard passes on a freeze-forever run
class _Toy(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        return {'image': torch.rand(3, 16, 16), 'mask': (torch.rand(16, 16) > 0.5).long(), 'sample_id': str(i)}

class _EncDec(nn.Module):  # stub with .encoder so the drift check targets real params
    def __init__(self):
        super().__init__()
        self.encoder = nn.Conv2d(3, 4, 3, padding=1)
        self.decoder = nn.Conv2d(4, 4, 3, padding=1)
        self.segmentation_head = nn.Conv2d(4, 1, 3, padding=1)
    def forward(self, x): return self.segmentation_head(self.decoder(self.encoder(x)))

loader = DataLoader(_Toy(), batch_size=2)
with tempfile.TemporaryDirectory() as d:
    cbs = build_default_callbacks(d, activation_stats=False, pred_overlays=False,
                                  drift_check=True, drift_expect_frozen=True)
    assert len(cbs) == 1 and isinstance(cbs[-1], EncoderDriftCheck)
    train(root=None, model=_EncDec(), train_loader=loader, val_loader=loader,
          num_epochs=1, batch_size=2, freeze_epochs=-1, device_str='cpu',  # frozen forever -> must pass
          checkpoint_dir=d, use_amp=False, resume=False, callbacks=cbs)


In [ ]:
# smoke test: EncoderDriftCheck — drifted encoder raises under expect_frozen, reports otherwise; no-.encoder models are a no-op
loader = DataLoader(_Toy(), batch_size=2)
with tempfile.TemporaryDirectory() as d:
    guard = EncoderDriftCheck(expect_frozen=True)
    raised = False
    try:
        train(root=None, model=_EncDec(), train_loader=loader, val_loader=loader,
              num_epochs=1, batch_size=2, freeze_epochs=0, device_str='cpu',  # encoder trains -> drifts
              checkpoint_dir=d, use_amp=False, resume=False, callbacks=[guard])
    except RuntimeError as e:
        raised = 'encoder drifted' in str(e)
    assert raised, 'expected RuntimeError for a drifted encoder with expect_frozen=True'
    rep = EncoderDriftCheck(expect_frozen=False)
    train(root=None, model=_EncDec(), train_loader=loader, val_loader=loader,
          num_epochs=1, batch_size=2, freeze_epochs=0, device_str='cpu',
          checkpoint_dir=d, use_amp=False, resume=False, callbacks=[rep])  # report-only: no raise

chk = EncoderDriftCheck(expect_frozen=True)  # no-.encoder model: guard is a no-op
st = TrainState(model=nn.Conv2d(3, 1, 3, padding=1), device=torch.device('cpu'), num_epochs=1)
chk.on_train_begin(st); chk.on_train_end(st)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()